# Análisis de inscripciones

Este notebook corresponde a la Parte 2 de la prueba técnica.

El objetivo es conectarse a la base de datos MariaDB, cargar las tablas principales en DataFrames de Pandas y realizar análisis sobre alumnos, programas, inscripciones e historial de cambios de estatus.

## 1. Importación de librerías

In [1]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from getpass import getpass

## 2. Configuración de conexión a MariaDB

La conexión se configura con variables simples para que pueda adaptarse fácilmente a otro entorno local.
La contraseña se solicita en tiempo de ejecución para evitar dejarla escrita dentro del notebook.

In [5]:
DB_USER = "python_user"
DB_HOST = "localhost"
DB_PORT = 3306
DB_NAME = "inscripciones_db"

DB_PASSWORD = getpass("Contraseña de MariaDB: ")

In [7]:
connection_url = URL.create(
    drivername="mysql+pymysql",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)

engine = create_engine(connection_url)

## 3. Carga de tablas en DataFrames

In [8]:
df_alumnos = pd.read_sql("SELECT * FROM alumnos", engine)
df_programas = pd.read_sql("SELECT * FROM programas", engine)
df_inscripciones = pd.read_sql("SELECT * FROM inscripciones", engine)
df_historial = pd.read_sql("SELECT * FROM historial_estatus", engine)

## 4. Verificación inicial de datos cargados

In [9]:
print("Alumnos:", df_alumnos.shape)
print("Programas:", df_programas.shape)
print("Inscripciones:", df_inscripciones.shape)
print("Historial:", df_historial.shape)

Alumnos: (465, 4)
Programas: (10, 2)
Inscripciones: (465, 5)
Historial: (15, 6)


In [10]:
display(df_alumnos.head())
display(df_programas.head())
display(df_inscripciones.head())
display(df_historial.head())

,id_alumno,nombre,empresa,fecha_ingreso
0,1001,Luis Cruz,Soriana,2025-02-26
1,1002,Andrés Ramírez,Coppel,2022-10-04
2,1003,Daniela Ortiz,Soriana,2017-10-02
3,1004,Luis Reyes,Soriana,2015-06-06
4,1005,Sofía Hernández,Soriana,2017-03-31


,id_programa,nombre_programa
0,8,Bachillerato Ejecutivo
1,10,Bachillerato General
2,5,Lic. Administración
3,2,Lic. Contaduría
4,3,Lic. Logística


,id_inscripcion,id_alumno,id_programa,estatus_actual,fecha_inscripcion
0,1,1001,1,suspendido,2025-02-26
1,2,1002,2,suspendido,2022-10-04
2,3,1003,3,egresado,2017-10-02
3,4,1004,3,egresado,2015-06-06
4,5,1005,3,baja_programa,2017-03-31


,id_historial,id_inscripcion,estatus_anterior,estatus_nuevo,fecha_cambio,motivo
0,1,1,inscrito,activo,2026-02-28,Inicio regular del programa
1,2,1,activo,baja_empresa,2026-06-03,Cambio laboral reportado por la empresa
2,3,1,baja_empresa,activo,2026-06-23,Reingreso autorizado por reincorporación laboral
3,4,7,inscrito,activo,2026-03-30,Activación inicial de inscripción
4,5,7,activo,baja_empresa,2026-06-16,Finalización de relación laboral con la empresa


## 5. Construcción del DataFrame consolidado

Se construye un DataFrame principal combinando alumnos, inscripciones y programas.

Este DataFrame será la base para los análisis de distribución de estatus, tasas por programa y evolución de inscripciones.

In [11]:
df_base = (
    df_inscripciones
    .merge(df_alumnos, on="id_alumno", how="inner")
    .merge(df_programas, on="id_programa", how="inner")
)

df_base.head()

,id_inscripcion,id_alumno,id_programa,estatus_actual,fecha_inscripcion,nombre,empresa,fecha_ingreso,nombre_programa
0,1,1001,1,suspendido,2025-02-26,Luis Cruz,Soriana,2025-02-26,Lic. Negocios
1,2,1002,2,suspendido,2022-10-04,Andrés Ramírez,Coppel,2022-10-04,Lic. Contaduría
2,3,1003,3,egresado,2017-10-02,Daniela Ortiz,Soriana,2017-10-02,Lic. Logística
3,4,1004,3,egresado,2015-06-06,Luis Reyes,Soriana,2015-06-06,Lic. Logística
4,5,1005,3,baja_programa,2017-03-31,Sofía Hernández,Soriana,2017-03-31,Lic. Logística


## 6. Revisión de estructura del DataFrame base

In [12]:
print("Filas y columnas:", df_base.shape)

df_base.info()

Filas y columnas: (465, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 465 entries, 0 to 464
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_inscripcion     465 non-null    int64 
 1   id_alumno          465 non-null    int64 
 2   id_programa        465 non-null    int64 
 3   estatus_actual     465 non-null    object
 4   fecha_inscripcion  465 non-null    object
 5   nombre             465 non-null    object
 6   empresa            465 non-null    object
 7   fecha_ingreso      465 non-null    object
 8   nombre_programa    465 non-null    object
dtypes: int64(3), object(6)
memory usage: 32.8+ KB


In [13]:
df_base[[
    "id_inscripcion",
    "id_alumno",
    "nombre",
    "empresa",
    "nombre_programa",
    "estatus_actual",
    "fecha_ingreso",
    "fecha_inscripcion"
]].head(10)

,id_inscripcion,id_alumno,nombre,empresa,nombre_programa,estatus_actual,fecha_ingreso,fecha_inscripcion
0,1,1001,Luis Cruz,Soriana,Lic. Negocios,suspendido,2025-02-26,2025-02-26
1,2,1002,Andrés Ramírez,Coppel,Lic. Contaduría,suspendido,2022-10-04,2022-10-04
2,3,1003,Daniela Ortiz,Soriana,Lic. Logística,egresado,2017-10-02,2017-10-02
3,4,1004,Luis Reyes,Soriana,Lic. Logística,egresado,2015-06-06,2015-06-06
4,5,1005,Sofía Hernández,Soriana,Lic. Logística,baja_programa,2017-03-31,2017-03-31
5,6,1006,Paola Mendoza,Bayer,Secundaria Abierta,baja_programa,2023-03-28,2023-03-28
6,7,1007,Gabriela Cruz,Soriana,Lic. Negocios,baja_empresa,2024-03-30,2024-03-30
7,8,1008,Gabriela Morales,Soriana,Lic. Negocios,baja_empresa,2018-07-05,2018-07-05
8,9,1009,Paola López,Soriana,Lic. Logística,egresado,2017-06-01,2017-06-01
9,10,1010,Valeria Torres,Soriana,Lic. Logística,egresado,2020-11-04,2020-11-04


## 7. Validación de consistencia

In [14]:
print("Inscripciones originales:", df_inscripciones.shape[0])
print("Inscripciones en df_base:", df_base.shape[0])

if df_inscripciones.shape[0] == df_base.shape[0]:
    print("Validación correcta: no se perdieron ni duplicaron inscripciones.")
else:
    print("Revisar joins: la cantidad de inscripciones no coincide.")

Inscripciones originales: 465
Inscripciones en df_base: 465
Validación correcta: no se perdieron ni duplicaron inscripciones.


## 8. Conversión de fechas

In [15]:
df_base["fecha_ingreso"] = pd.to_datetime(df_base["fecha_ingreso"])
df_base["fecha_inscripcion"] = pd.to_datetime(df_base["fecha_inscripcion"])

df_historial["fecha_cambio"] = pd.to_datetime(df_historial["fecha_cambio"])

print("Tipos de datos en df_base:")
display(df_base.dtypes)

print("Tipos de datos en df_historial:")
display(df_historial.dtypes)

Tipos de datos en df_base:


id_inscripcion                int64
id_alumno                     int64
id_programa                   int64
estatus_actual               object
fecha_inscripcion    datetime64[ns]
nombre                       object
empresa                      object
fecha_ingreso        datetime64[ns]
nombre_programa              object
dtype: object

Tipos de datos en df_historial:


id_historial                 int64
id_inscripcion               int64
estatus_anterior            object
estatus_nuevo               object
fecha_cambio        datetime64[ns]
motivo                      object
dtype: object